## Deterministic references and comparisons

This section can be run on its own in a fresh kernel; it does not rerun the long
Monte Carlo experiments. The numerical routines are in `src/msn_p8/references.py`.
Comparisons use summaries retained by a full notebook run, or the **saved printed
MC outputs** if those experiments have not been run in the current kernel.
Save new MC outputs before using the latter fallback.

### Classical integration and quantum spectrum

Write $y=x/a$, $\theta=k_BT/V_0$ and $v(y)=(y^2-1)^2$. At $P=1$,
$$\rho_{\mathrm{cl}}(y)=\frac{e^{-v(y)/\theta}}{\int e^{-v(y)/\theta}\,dy},
\qquad \langle v\rangle_{\mathrm{cl}}=\int v(y)\rho_{\mathrm{cl}}(y)\,dy.$$
We reproduce Question 3's 2,001-point trapezoidal integral on $[-2.5,2.5]$.

For the quantum limit, solve $[-C\partial_y^2+v(y)]\psi_n=\varepsilon_n\psi_n$.
A centered finite difference on a grid of spacing $h$ gives diagonal entries
$2C/h^2+v(y_i)$ and off-diagonal entries $-C/h^2$, with zero wavefunctions at
$y=\pm4$. The first 32 states on 4,001 grid points give
$$w_n=\frac{e^{-(\varepsilon_n-\varepsilon_0)/\theta}}
{\sum_m e^{-(\varepsilon_m-\varepsilon_0)/\theta}},\quad
\rho_q(y)=\sum_n w_n|\psi_n(y)|^2,\quad
\langle v\rangle_q=\sum_n w_n\int v(y)|\psi_n(y)|^2\,dy.$$
The ground-state **total** energy $\varepsilon_0$ is different from the
**potential** energy $\langle v\rangle_q$ measured by the MC estimator.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants as ref_constants
from IPython.display import display
from msn_p8.references import (
    classical_reference, quantum_reference, transfer_reference, saved_mc_results,
)

reference_dir = Path("output/report")
reference_dir.mkdir(parents=True, exist_ok=True)
reference_cases = [(0.3, 0.4), (0.3, 0.6), (0.9, 0.4), (0.9, 0.6)]
reference_classical = classical_reference()
reference_quantum = {case: quantum_reference(*case) for case in reference_cases}
print(f"Classical <V>/V0 at theta=0.4: {reference_classical['potential_energy']:.9f}")
print(" C   theta     E0/V0      <V>/V0       V0 (eV)     T (K)")
for (C, theta), result in reference_quantum.items():
    V0_ref = ref_constants.hbar**2/(2*ref_constants.m_p*(0.04e-9)**2*C)
    print(f"{C:.1f}   {theta:.1f}    {result['ground_energy']:.6f}    "
          f"{result['potential_energy']:.6f}     {V0_ref/ref_constants.e:.6f}    "
          f"{theta*V0_ref/ref_constants.k:.2f}")

# Check grid spacing, finite domain, and truncation of the thermal spectrum.
reference_checks = {}
classical_refined = classical_reference(half_width=3.0, points=4001)
reference_checks['classical_refinement'] = abs(
    classical_refined['potential_energy'] - reference_classical['potential_energy'])
for case, result in reference_quantum.items():
    variants = [quantum_reference(*case, points=8001),
                quantum_reference(*case, half_width=4.5, points=4501),
                quantum_reference(*case, states=48)]
    change = max(abs(r['potential_energy']-result['potential_energy']) for r in variants)
    reference_checks[f'quantum_{case}'] = change
    assert change < 2e-6, (case, change)
    assert result['last_state_weight'] < 1e-12
print("Largest energy change under refinement:", f"{max(reference_checks.values()):.2e} V0")

fig, ax = plt.subplots(figsize=(7.2, 3.0), constrained_layout=True)
plot_y = np.linspace(-1.65, 1.65, 600)
ax.plot(plot_y, (plot_y**2-1)**2, color='black')
ax.set(xlabel=r"Reduced position $y=x/a$", ylabel=r"$V(x)/V_0$", ylim=(0, 3))
ax.grid(alpha=.25)
fig.savefig(reference_dir / 'double_well.pdf')
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.0), constrained_layout=True)
for ax, case in zip(axes.flat, reference_cases):
    result = reference_quantum[case]
    classical = classical_reference(theta=case[1])
    ax.plot(classical['y'], classical['density'], '--', color='gray', label='Classical')
    ax.plot(result['y'], result['density'], color='tab:blue', label='Quantum limit')
    ax.set(title=f'C={case[0]}, theta={case[1]}', xlim=(-2,2),
           xlabel=r'$y=x/a$', ylabel='Density in y')
    ax.grid(alpha=.25)
axes[0,0].legend()
fig.savefig(reference_dir / 'quantum_densities.pdf')
display(fig)
plt.close(fig)

### Integrating the same finite-$P$ path integral

The finite-difference spectrum benchmarks the quantum limit, whereas a transfer
matrix tests the exact same finite-$P$ distribution sampled by Metropolis.
On a quadrature grid of spacing $h$, define
$$K_{ij}=h\exp\left[-\frac{P\theta}{4C}(y_i-y_j)^2
-\frac{v(y_i)+v(y_j)}{2P\theta}\right].$$
The free-particle prefactor cancels in normalized averages. Then
$\rho_P(y_i)=(K^P)_{ii}/[h\,\mathrm{Tr}(K^P)]$ and
$\langle v\rangle_P=\sum_i h\,v(y_i)\rho_P(y_i)$.

The centroid cannot be obtained by averaging the one-point quantum density.
With $D(k)_{ii}=e^{iky_i/P}$ its characteristic function is
$$\chi_P(k)=\frac{\mathrm{Tr}[(D(k)K)^P]}{\mathrm{Tr}(K^P)},\qquad
\rho_{c,P}(q)=\frac{1}{2\pi}\int e^{-ikq}\chi_P(k)\,dk.$$
We use 141 quadrature points on $[-3.5,3.5]$, 80 positive Fourier modes with
period 6.4, and analytic Fourier integration across the MC bins. This gives
the **same central-bin estimator** as the simulations: bin $[0,0.05]$ divided
by the maximum bin density. It approximates, rather than exactly equals,
$-\theta\log[\rho_c(0)/\rho_{c,\max}]$.

$P=15$ separates MC sampling error from discretization error; $P=60$ checks
the approach towards the quantum path-integral limit. These are deterministic
numerical references, not closed-form exact solutions. Grid, Fourier-cutoff,
and finite-$P$ refinements are checked below.

In [ ]:
reference_edges = np.linspace(-2.5, 2.5, 101)
reference_p15 = {case: transfer_reference(*case, 15, edges=reference_edges)
                 for case in reference_cases}
reference_p60 = {case: transfer_reference(*case, 60, edges=reference_edges)
                 for case in reference_cases}
print(" C  theta    <V>_P15/V0   quantum <V>/V0    P15 bias (%)    barrier P15    barrier P60")
for case in reference_cases:
    v15 = reference_p15[case]['potential_energy']
    vq = reference_quantum[case]['potential_energy']
    print(f"{case[0]:.1f}  {case[1]:.1f}      {v15:.6f}        {vq:.6f}        "
          f"{100*(v15/vq-1):+.3f}          {reference_p15[case]['barrier']:.6f}       "
          f"{reference_p60[case]['barrier']:.6f}")

for case in reference_cases:
    original = reference_p15[case]
    variants = [transfer_reference(*case, 15, grid_points=281),
                transfer_reference(*case, 15, modes=100),
                transfer_reference(*case, 15, period=8.0, modes=100),
                transfer_reference(*case, 15, half_width=4.0, grid_points=161)]
    change = max(abs(r['barrier']-original['barrier']) for r in variants)
    reference_checks[f'centroid_{case}'] = change
    assert change < 2e-5, (case, change)
    assert original['fourier_tail'] < 1e-10
    assert original['centroid_density'].min() > -1e-10
    integral = np.mean(original['centroid_density'])*6.4
    assert abs(integral-1) < 1e-10
print('Largest P15 barrier change under quadrature/Fourier refinement:',
      f"{max(value for key,value in reference_checks.items() if key.startswith('centroid')):.2e} V0")

reference_P = np.array([1, 2, 3, 4, 6, 8, 12, 15, 20, 30, 40, 60])
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.6), constrained_layout=True)
for ax, C in zip(axes, [0.3, 0.9]):
    vq = reference_quantum[(C, .4)]['potential_energy']
    values = np.array([transfer_reference(C, .4, int(P), grid_points=281,
                      include_centroid=False)['potential_energy'] for P in reference_P])
    ax.plot(reference_P, values, 'o-', label='Finite-P integral')
    ax.axhline(vq, color='black', ls='--', label='Quantum reference')
    ax.axhspan(.99*vq, 1.01*vq, alpha=.15, color='green', label='1% band')
    ax.set(title=f'C={C}, theta=0.4', xlabel='Number of beads P', ylabel=r'$\langle V\rangle/V_0$')
    ax.grid(alpha=.25)
axes[0].legend(fontsize=8)
fig.savefig(reference_dir / 'reference_energy_convergence.pdf')
display(fig)
plt.close(fig)

### Comparison with the measured Monte Carlo estimates

At the lower temperature the MC barrier comes from pooling three centroid
densities before taking the logarithm. Its displayed spread is the standard
deviation of three individual barrier estimates, not a standard error or a
confidence interval. At the higher temperature the existing experiments use
only seed 20, so no corresponding spread is available. All MC cases use $P=15$.
The reference below matches the bins and finite $P$; increasing $P$ cannot by
itself correct inadequate sampling of that finite-$P$ distribution.

In [ ]:
reference_mc = dict(globals().get('mc_comparisons', {}))
saved_comparisons = saved_mc_results(Path('main.ipynb'))
for case in reference_cases:
    if case not in reference_mc and case in saved_comparisons:
        reference_mc[case] = saved_comparisons[case]
print('MC summaries: current-kernel records where available; otherwise saved printed outputs.')
print(' C  theta     MC barrier    run-to-run SD    runs    matched P15 reference')
for case in reference_cases:
    measured = reference_mc.get(case)
    if measured is None:
        print(case, 'MC result not available')
        continue
    spread = 'not estimated' if measured['sd'] is None else f"{measured['sd']:.3f}"
    print(f"{case[0]:.1f}  {case[1]:.1f}       {measured['barrier']:.4f}       "
          f"{spread:>13}      {measured['runs']}         {reference_p15[case]['barrier']:.6f}")

fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.0), constrained_layout=True)
reference_centers = (reference_edges[:-1]+reference_edges[1:])/2
for ax, case in zip(axes.flat, reference_cases):
    for P, collection, style in [(15, reference_p15, '-'), (60, reference_p60, '--')]:
        density = collection[case]['centroid_bins']
        valid = density > 1e-12
        free = np.full_like(density, np.nan)
        free[valid] = -case[1]*np.log(density[valid]/density.max())
        ax.plot(reference_centers, free, style, label=f'Deterministic P={P}')
    ax.plot(reference_centers, (reference_centers**2-1)**2, ':', color='gray', label='Bare potential')
    if case in reference_mc:
        measured = reference_mc[case]
        ax.plot(.025, measured['barrier'], 'o', color='tab:red', label='Saved MC estimate')
    ax.set(title=f'C={case[0]}, theta={case[1]}', xlim=(-1.5,1.5), ylim=(0,1.2),
           xlabel=r'$y=x/a$', ylabel=r'$F_c/V_0$')
    ax.grid(alpha=.25)
axes[0,0].legend(fontsize=7)
fig.savefig(reference_dir / 'reference_barriers.pdf')
display(fig)
plt.close(fig)

# Machine-readable values used by the report, regenerated by this section.
reference_summary = {
    'classical': reference_classical['potential_energy'], 'checks': reference_checks,
    'cases': [{
        'C': C, 'theta': theta,
        'ground_energy': reference_quantum[(C,theta)]['ground_energy'],
        'quantum_potential': reference_quantum[(C,theta)]['potential_energy'],
        'P15_potential': reference_p15[(C,theta)]['potential_energy'],
        'P15_barrier': reference_p15[(C,theta)]['barrier'],
        'P60_barrier': reference_p60[(C,theta)]['barrier'],
        'mc': reference_mc.get((C,theta)),
    } for C,theta in reference_cases],
}
(reference_dir/'reference_results.json').write_text(
    json.dumps(reference_summary, indent=2), encoding='utf-8')